In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

# Enable tqdm progress visualization for pandas operations
tqdm.pandas()

# Define file paths
DATASET_PATH = r"C:\Users\User\Desktop\folders\python\projects\medicine_dataset\A_Z_medicines_dataset_with_distance.csv"
MODEL_OUTPUT_PATH = r"C:\Users\User\Desktop\folders\python\projects\medicine_dataset\medicine_tfidf_model.joblib"
DB_OUTPUT_PATH = r"C:\Users\User\Desktop\folders\python\projects\medicine_dataset\processed_medicines_db.joblib"


def parse_pack_label(label):
    if not isinstance(label, str):
        return 'other', 1

    qty_match = re.search(r'\b\d+\b', label)
    quantity = int(qty_match.group(0)) if qty_match else 1

    lbl = label.lower()
    if 'tablet' in lbl:
        form = 'tablet'
    elif 'syrup' in lbl:
        form = 'syrup'
    elif 'inject' in lbl:
        form = 'injection'
    elif 'capsule' in lbl:
        form = 'capsule'
    else:
        form = 'other'

    return form, quantity


def preprocess_dataset(df):
    data = df.copy()

    print("[1/4] Handling missing values...")
    data['short_composition2'] = data['short_composition2'].fillna('None')

    print("[2/4] Parsing packaging labels (Regex Extraction)...")
    # Using progress_apply to display a live progress bar over 250k rows
    parsed_info = data['pack_size_label'].progress_apply(parse_pack_label)
    data['form'], data['quantity'] = zip(*parsed_info)

    print("[3/4] Normalizing unit pricing...")
    data['price_per_unit'] = data['price(₹)'] / data['quantity']

    print("[4/4] Building text corpus...")
    data['text_corpus'] = (
        data['manufacturer_name'] + " " + data['pack_size_label']
    )

    return data


def train_and_save():
    print(f"=== STEP 1: Loading Dataset ===")
    print(f"Path: {DATASET_PATH}")
    df_raw = pd.read_csv(DATASET_PATH)
    print(f"Dataset successfully loaded! Total rows: {len(df_raw):,}\n")

    print(f"=== STEP 2: Preprocessing & Feature Engineering ===")
    df_clean = preprocess_dataset(df_raw)
    print("Preprocessing complete!\n")

    print(f"=== STEP 3: Fitting Machine Learning Model ===")
    print("Fitting TF-IDF Vectorizer on text features...")
    tfidf_model = TfidfVectorizer(stop_words='english', sublinear_tf=True)
    
    # Fit the vectorizer
    tfidf_model.fit(df_clean['text_corpus'])
    print("TF-IDF Vectorizer fitting complete!\n")

    print(f"=== STEP 4: Exporting Model Artifacts ===")
    print("Saving TF-IDF Model (.joblib)...")
    joblib.dump(tfidf_model, MODEL_OUTPUT_PATH, compress=3)
    
    print("Saving Processed Dataset (.joblib)...")
    joblib.dump(df_clean, DB_OUTPUT_PATH, compress=3)

    print("\n==========================================")
    print("        TRAINING SUCCESSFUL!             ")
    print("==========================================")
    print(f"Model File:    {MODEL_OUTPUT_PATH}")
    print(f"Database File: {DB_OUTPUT_PATH}")


if __name__ == "__main__":
    train_and_save()